In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split

# 1. Charger ton fichier de données (adapte le chemin si ton notebook est dans le dossier 'notebooks/')
df = pd.read_csv('/home/yassine/MedRoute-AI/data/raw/diabetes/diabetes.csv') # Remplace par le nom exact de ton fichier CSV si besoin

# 2. Séparer les variables explicatives (X) et la cible (y)
# (Assure-toi que 'Outcome' ou 'target' correspond bien au nom de ta colonne cible)
target_column = 'Outcome' if 'Outcome' in df.columns else df.columns[-1]
X = df.drop(columns=[target_column])
y = df[target_column]

# 3. Diviser les données en train et test (80% / 20%)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Données chargées ! X_train shape: {X_train.shape}, X_test shape: {X_test.shape}")

Données chargées ! X_train shape: (614, 8), X_test shape: (154, 8)


In [3]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import joblib

def optimize_random_forest(X_train, y_train, X_test, y_test):
    param_dist = {
        'n_estimators': [50, 100, 200, 300, 500],
        'max_depth': [None, 10, 20, 30, 40],
        'min_samples_split': [2, 5, 10],
        'min_samples_leaf': [1, 2, 4],
        'bootstrap': [True, False],
        'class_weight': ['balanced', 'balanced_subsample', None]
    }

    rf = RandomForestClassifier(random_state=42)

    random_search = RandomizedSearchCV(
        estimator=rf,
        param_distributions=param_dist,
        n_iter=20,
        cv=5,
        scoring='f1',
        random_state=42,
        n_jobs=-1,
        verbose=1
    )

    print("Recherche des meilleurs hyperparamètres en cours...")
    random_search.fit(X_train, y_train)

    best_rf = random_search.best_estimator_
    print(f"\nMeilleurs paramètres trouvés :\n{random_search.best_params_}")

    y_pred = best_rf.predict(X_test)
    
    print("\n--- Évaluation du Modèle Optimisé ---")
    print(f"Accuracy : {accuracy_score(y_test, y_pred):.4f}")
    print("\nMatrice de confusion :")
    print(confusion_matrix(y_test, y_pred))
    print("\nRapport de classification :")
    print(classification_report(y_test, y_pred))

    # Sauvegarde dans le dossier models/ à la racine
    joblib.dump(best_rf, "../models/diabetes_random_forest_optimized.pkl")
    print("\nModèle optimisé sauvegardé avec succès dans models/ !")

    return best_rf

In [4]:
# Lancer l'optimisation et afficher l'évaluation
best_rf_model = optimize_random_forest(X_train, y_train, X_test, y_test)

Recherche des meilleurs hyperparamètres en cours...
Fitting 5 folds for each of 20 candidates, totalling 100 fits

Meilleurs paramètres trouvés :
{'n_estimators': 300, 'min_samples_split': 10, 'min_samples_leaf': 4, 'max_depth': None, 'class_weight': 'balanced', 'bootstrap': True}

--- Évaluation du Modèle Optimisé ---
Accuracy : 0.7792

Matrice de confusion :
[[77 22]
 [12 43]]

Rapport de classification :
              precision    recall  f1-score   support

           0       0.87      0.78      0.82        99
           1       0.66      0.78      0.72        55

    accuracy                           0.78       154
   macro avg       0.76      0.78      0.77       154
weighted avg       0.79      0.78      0.78       154


Modèle optimisé sauvegardé avec succès dans models/ !
